# Clinical fine-tune — Model 1

Four cells. Run them **after** the training notebook's own cells (0–8), or paste them
into that notebook after the self-training cell.

They fine-tune the segmenter on 89 wound outlines drawn by hand across 31 wounds and
two hospitals, then **judge the result in centimetres**, the way the app measures, and
refuse to save anything that fails.

Why: the model reaches a good Dice but is ~26% away from the clinician in centimetres,
and the error is polarised by wound type. Outlining the same photographs by hand and
measuring both the same way reaches ~12%. Only the boundary differs.

## 1 · Load the outlined wounds

Runs on its own. Everything the later cells need is checked here **by name**, so a
missing piece is reported rather than raising `NameError` three cells later.

In [ ]:
# ============================================================
# CLINICAL 1/4 — load the hand-outlined wounds
# ============================================================
# Runs on its own. Everything the later cells need from the notebook is checked
# here by name, so a missing piece says which one instead of raising NameError
# five cells later.
import json, os
import numpy as np, cv2, tensorflow as tf

IMG_H = globals().get("IMG_HEIGHT", 384)
IMG_W = globals().get("IMG_WIDTH", 384)
print(f"input size: {IMG_H}x{IMG_W}"
      + ("" if "IMG_HEIGHT" in globals() else "   (notebook value not found, using 384)"))


def _find_clinical():
    for c in ("/kaggle/input/datasets/ibrahimshehada/diafootcare-wound-outlines",
              "/kaggle/input/diafootcare-wound-outlines"):
        if os.path.exists(os.path.join(c, "index.json")):
            return c
    for root, _dirs, files in os.walk("/kaggle/input"):
        if "index.json" in files and "masks" in _dirs:
            return root
    return None


CLIN_DIR = _find_clinical()
assert CLIN_DIR, "attach the diafootcare-wound-outlines dataset first"
print("clinical set:", CLIN_DIR)

_index = json.load(open(os.path.join(CLIN_DIR, "index.json"), encoding="utf-8"))
_splits = json.load(open(os.path.join(CLIN_DIR, "splits.json"), encoding="utf-8"))
_gate_wounds = list(json.load(open(os.path.join(CLIN_DIR, "gate.json"), encoding="utf-8")))
clin_tr = [it for it in _index if it["wound"] in _splits["train"]]
clin_va = [it for it in _index if it["wound"] in _splits["val"]]


def _clin_pair(it):
    img = cv2.cvtColor(cv2.imread(os.path.join(CLIN_DIR, "images", it["file"])), cv2.COLOR_BGR2RGB)
    msk = cv2.imread(os.path.join(CLIN_DIR, "masks", it["file"]), cv2.IMREAD_GRAYSCALE)
    if img.shape[0] != IMG_H or img.shape[1] != IMG_W:
        img = cv2.resize(img, (IMG_W, IMG_H))
        msk = cv2.resize(msk, (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    return img.astype(np.float32) / 255.0, (msk > 127).astype(np.float32)[..., None]


Xc_tr = np.array([_clin_pair(it)[0] for it in clin_tr], np.float32)
Yc_tr = np.array([_clin_pair(it)[1] for it in clin_tr], np.float32)
Xc_va = np.array([_clin_pair(it)[0] for it in clin_va], np.float32)
Yc_va = np.array([_clin_pair(it)[1] for it in clin_va], np.float32)

print(f"train {len(Xc_tr)} photographs / {len(_splits['train'])} wounds")
print(f"val   {len(Xc_va)} photographs / {len(_splits['val'])} wounds   "
      f"(split BY WOUND — three shots of one ulcer are nearly the same picture)")
print(f"gate  {len(_gate_wounds)} wounds the current model already measures well")

_missing = [n for n in ("model", "Xp_os", "Yp_os", "make_ds", "make_opt",
                        "make_callbacks", "seg_loss", "dice_coef", "iou_metric",
                        "val_ds", "FINE_TUNE_FROM", "BatchNormalization")
            if n not in globals()]
if _missing:
    print("\n⚠️  not in memory yet: " + ", ".join(_missing))
    print("    Run the notebook's own cells first (Run All). Cells 3 and 4 below")
    print("    train and gate; they need the model and the precise pool from above.")
else:
    print("\nall notebook pieces present — cells 2, 3, 4 can run")


## 2 · Measure the way the app measures

In **centimetres**, not Dice. A mask can gain Dice while losing the extent that
actually gets measured, and centimetres are what the clinician is handed.

The label guard is part of it: on 40% of small-label photographs the segmenter reads
the printed magenta ring as granulation and returns the ring's own diameter as the
wound size.

In [ ]:
# ============================================================
# CLINICAL 2/4 — measure the model the way the app measures it
# ============================================================
# In CENTIMETRES, not Dice. A mask can gain Dice while losing the extent that
# actually gets measured, and centimetres are what the clinician is handed.
# Every step below runs in ai_service.dart, the label guard included: on 40% of
# small-label photographs the segmenter reads the printed magenta ring as
# granulation and returns the ring's own diameter as the wound size.
import numpy as np, cv2, os


def _measure_cm(prob, item):
    m = (prob >= 0.5).astype(np.uint8)
    if m.sum() == 0:                                  # 0.5x-peak fallback
        pk = float(prob.max())
        if pk <= 0:
            return None
        m = (prob > 0.5 * pk).astype(np.uint8)
    k = np.ones((5, 5), np.uint8)
    m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, k), cv2.MORPH_CLOSE, k)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return None

    bgr = cv2.imread(os.path.join(CLIN_DIR, "images", item["file"]))
    hsv = cv2.cvtColor(cv2.resize(bgr, (m.shape[1], m.shape[0])), cv2.COLOR_BGR2HSV)
    paper = (hsv[..., 1] < 50) & (hsv[..., 2] > 170)  # printed ink sits on a white card
    keep = []
    for i in range(1, n):
        if st[i, cv2.CC_STAT_AREA] < 10:
            continue
        comp = (lab == i).astype(np.uint8)
        collar = (cv2.dilate(comp, k, iterations=2) > 0) & (comp == 0)
        if collar.sum() and paper[collar].mean() >= 0.40:
            continue                                  # that blob is the label
        keep.append(i)
    if not keep:
        return None

    idx = max(keep, key=lambda i: st[i, cv2.CC_STAT_AREA])
    ys, xs = np.where(lab == idx)
    px, py = item.get("ppc_x"), item.get("ppc_y")     # pixels per cm, from the ring
    if not px or not py:
        return None
    sx, sy = m.shape[1] / IMG_W, m.shape[0] / IMG_H
    p = np.stack([xs * sx / px, ys * sy / py], 1).astype(np.float64)
    p -= p.mean(0)
    _, _, vt = np.linalg.svd(p, full_matrices=False)  # principal axis = longest extent
    q = p @ vt.T
    return float(q[:, 0].max() - q[:, 0].min())


def clinical_report(net, title):
    per, dices, unmeasured = {}, [], 0
    P = net.predict(Xc_va, batch_size=8, verbose=0)[..., 0]
    Pf = net.predict(Xc_va[:, :, ::-1, :], batch_size=8, verbose=0)[..., 0][:, :, ::-1]
    P = (P + Pf) / 2.0                                # 2-view TTA, as the app does
    for i, it in enumerate(clin_va):
        gt, pr = Yc_va[i, ..., 0] > 0.5, P[i] >= 0.5
        dices.append(2 * (gt & pr).sum() / max(1e-6, gt.sum() + pr.sum()))
        if it.get("ref_quality") != "measured" or not it.get("ref_major"):
            continue                                  # an estimate by eye is not truth
        cm = _measure_cm(P[i], it)
        if cm is None:
            unmeasured += 1
            continue
        per.setdefault(it["wound"], []).append(100.0 * (cm - it["ref_major"]) / it["ref_major"])
    w = {k: float(np.mean(np.abs(v))) for k, v in per.items()}
    print("\n--- " + title + " ---")
    print(f"  clinical Dice {np.mean(dices):.3f} | wounds scored: {len(w)} | "
          f"unmeasurable: {unmeasured}")
    for k, v in sorted(w.items(), key=lambda kv: -kv[1]):
        print(f"    {k:<42}{v:6.1f}%")
    if w:
        print(f"  MEAN ABSOLUTE ERROR: {np.mean(list(w.values())):.1f}%")
    return w, float(np.mean(dices))


fuseg_before = model.evaluate(val_ds, verbose=0, return_dict=True)["dice_coef"]
before, cdice_before = clinical_report(model, "BEFORE clinical fine-tune")
print(f"\nFUSeg val Dice (the original domain): {fuseg_before:.3f}")


## 3 · Fine-tune

Mixed with `Xp_os` — the precise FUSeg + DFUTissue pool the notebook already built.
89 photographs against ~1,270: ours alone would score well on our own split and
collapse everywhere else.

Decoder first, then a deep unfreeze at a low learning rate: adjust the boundary,
do not overwrite the model.

In [ ]:
# ============================================================
# CLINICAL 3/4 — fine-tune
# ============================================================
# Mixed with Xp_os, the precise FUSeg + DFUTissue pool this notebook already built
# and oversampled. 89 photographs against ~1,270: ours alone would score well on our
# own split and collapse everywhere else.
import numpy as np, tensorflow as tf, os

CLIN_OVERSAMPLE = 4
X_mix = np.concatenate([np.repeat(Xc_tr, CLIN_OVERSAMPLE, 0), Xp_os], 0)
Y_mix = np.concatenate([np.repeat(Yc_tr, CLIN_OVERSAMPLE, 0), Yp_os], 0)
print(f"mixed pool: {len(Xc_tr)}x{CLIN_OVERSAMPLE} clinical + {len(Xp_os)} precise "
      f"= {len(X_mix)}")

clin_tr_ds = make_ds(X_mix, Y_mix, training=True)
clin_va_ds = make_ds(Xc_va, Yc_va, training=False)

# Stage 1 — decoder only. A general encoder should not be moved by 89 photographs
# while the decoder is still adjusting to a new boundary convention.
model.backbone.trainable = False
model.compile(optimizer=make_opt(3e-4), loss=seg_loss,
              metrics=["accuracy", dice_coef, iou_metric])
print("\n[Clinical 1] decoder only, encoder frozen")
model.fit(clin_tr_ds, validation_data=clin_va_ds, epochs=12,
          callbacks=make_callbacks("unet_clinical.keras", patience=6))

# Stage 2 — the same deep unfreeze the notebook uses above, at a lower LR:
# adjust the boundary, do not overwrite the model.
model.backbone.trainable = True
for i, layer in enumerate(model.backbone.layers):
    if i < FINE_TUNE_FROM or isinstance(layer, BatchNormalization):
        layer.trainable = False
model.compile(optimizer=make_opt(2e-5), loss=seg_loss,
              metrics=["accuracy", dice_coef, iou_metric])
print("\n[Clinical 2] deep fine-tune, low LR")
model.fit(clin_tr_ds, validation_data=clin_va_ds, epochs=25,
          callbacks=make_callbacks("unet_clinical.keras", patience=10))

if os.path.exists("unet_clinical.keras"):
    model = tf.keras.models.load_model("unet_clinical.keras", compile=False)
    model.compile(optimizer="adam", loss=seg_loss, metrics=[dice_coef, iou_metric])
    print("\nloaded the best clinical checkpoint")


## 4 · The gate

Passes only if the clinical error improves, **no** wound the model already measured
well regresses, and FUSeg Dice has not fallen — forgetting the original domain fails
just as hard as failing to improve.

`unet_model.keras` is written **only on a pass**. On failure the previous model is
reloaded, so the export cell ships the old weights and a failed experiment leaves no
trace in what reaches a patient.

In [ ]:
# ============================================================
# CLINICAL 4/4 — the gate, then save only if it passes
# ============================================================
import json
import numpy as np, tensorflow as tf

after, cdice_after = clinical_report(model, "AFTER clinical fine-tune")
fuseg_after = model.evaluate(val_ds, verbose=0, return_dict=True)["dice_coef"]

print("\n" + "=" * 74 + "\nGATE\n" + "=" * 74)
mb = np.mean(list(before.values())) if before else float("nan")
ma = np.mean(list(after.values())) if after else float("nan")
print(f"  clinical cm error   {mb:6.1f}%  ->  {ma:6.1f}%")
print(f"  clinical Dice       {cdice_before:6.3f}  ->  {cdice_after:6.3f}")
print(f"  FUSeg val Dice      {fuseg_before:6.3f}  ->  {fuseg_after:6.3f}")

broke = []
print("\n  wounds the model already measured well:")
for w in _gate_wounds:
    # gate.json chooses WHICH wounds are checked. The baseline is this run's own
    # "before" pass, never a number from another script measured on other inputs.
    b, a = before.get(w), after.get(w)
    if b is None or a is None:
        print(f"    {w:<42} not scored this run")
        continue
    limit = max(b * 1.5, b + 5.0)          # tolerance, not a demand for perfection
    ok = a <= limit
    verdict = "ok" if ok else "REGRESSED"
    print(f"    {w:<42}{b:6.1f}% -> {a:6.1f}%   {verdict}")
    if not ok:
        broke.append(w)

forgot = fuseg_after < fuseg_before - 0.03
passed = (ma < mb) and not broke and not forgot

print("\n  " + ("PASS - unet_model.keras updated, the export cell will ship this"
                if passed else "FAIL - unet_model.keras left untouched"))
if not (ma < mb):
    print("    the clinical error did not improve")
for w in broke:
    print("    regressed: " + w)
if forgot:
    print(f"    forgot the original domain: FUSeg Dice {fuseg_before:.3f} -> {fuseg_after:.3f}")

json.dump(dict(before=before, after=after, clinical_dice=[cdice_before, cdice_after],
               fuseg_dice=[fuseg_before, fuseg_after], regressed=broke,
               forgot=bool(forgot), passed=bool(passed)),
          open("gate_report.json", "w"), indent=1)

if passed:
    model.save("unet_model.keras")
    print("\n  saved -> unet_model.keras")
else:
    # Reload the pre-fine-tune model so the export cell ships the OLD weights.
    # A failed experiment must leave no trace in what reaches a patient.
    model = tf.keras.models.load_model("unet_model.keras", compile=False)
    model.compile(optimizer="adam", loss=seg_loss, metrics=[dice_coef, iou_metric])
    print("\n  reloaded the pre-fine-tune model; nothing downstream changes.")
